In [21]:
import os
import cv2
import json
import joblib
import numpy as np
import mediapipe as mp
from collections import deque
import warnings
import pyautogui
from tensorflow.keras.models import load_model
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [22]:
MODEL_PATH = "data/hand_gesture_mlp.h5"
SCALER_PATH = "data/scaler.pkl"
ENCODER_PATH = "data/label_encoder.pkl"
GM_PATH = "data/gesture_map.json"

for path in [MODEL_PATH, SCALER_PATH, ENCODER_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ Không tìm thấy '{path}'. Hãy train mô hình trước!")

In [23]:
model = load_model(MODEL_PATH)          # Dùng Keras để load file .h5
scaler = joblib.load(SCALER_PATH)
le = joblib.load(ENCODER_PATH)
label_names = list(le.classes_)

In [24]:
LABEL_DESCRIPTIONS = {}
if os.path.exists(GM_PATH):
    try:
        with open(GM_PATH, "r", encoding="utf-8") as f:
            gm = json.load(f)
        for v in gm.values():
            if isinstance(v, dict) and "label" in v:
                LABEL_DESCRIPTIONS[v["label"]] = v.get("desc", "")
    except Exception:
        pass


In [25]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

In [26]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

True

In [27]:
SMOOTH_WIN = 5
UNKNOWN_THRESHOLD = 0.4
proba_buffer = deque(maxlen=SMOOTH_WIN)

In [28]:
# === CẤU HÌNH ĐIỀU KHIỂN CỬ CHỈ ===
import time

# Ánh xạ cử chỉ -> hành động
GESTURE_ACTIONS = {
    "thumb up": "volume_up",
    "fist": "volume_down",
    "peace": "scroll_up",
    "ok": "scroll_down",
    "palm": "pause_play",
    "index up": "left_click",
    "call": "right_click",
    "rock": "next_track",
    "gun sign": "prev_track",
    "L sign": "screenshot",
    # "pinch": "minimize_window",
    "C sign": "copy",
    "cross fingers": "alt_tab",
    "3 fingers up": "brightness_up",
}

# Thời gian cooldown giữa các hành động (giây)
ACTION_COOLDOWN = 1.0
last_action_time = {}
action_enabled = True  # Bật/tắt điều khiển bằng cử chỉ (nhấn 'c' để toggle)

In [29]:
def perform_action(gesture_name):
    """Thực hiện hành động điều khiển dựa trên cử chỉ"""
    global last_action_time, action_enabled
    
    if not action_enabled:
        return "Disabled", (128, 128, 128)
    
    # Kiểm tra cooldown
    current_time = time.time()
    if gesture_name in last_action_time:
        if current_time - last_action_time[gesture_name] < ACTION_COOLDOWN:
            return "Cooldown", (255, 165, 0)
    
    # Lấy hành động tương ứng
    action = GESTURE_ACTIONS.get(gesture_name)
    if not action:
        return "No Action", (128, 128, 128)
    
    try:
        # Thực hiện hành động
        if action == "volume_up":
            pyautogui.press('volumeup')
            last_action_time[gesture_name] = current_time
            return "🔊 Volume Up", (0, 255, 0)
        
        elif action == "volume_down":
            pyautogui.press('volumedown')
            last_action_time[gesture_name] = current_time
            return "🔉 Volume Down", (0, 255, 0)
        
        elif action == "scroll_up":
            pyautogui.scroll(300)
            last_action_time[gesture_name] = current_time
            return "⬆️ Scroll Up", (0, 255, 0)
        
        elif action == "scroll_down":
            pyautogui.scroll(-300)
            last_action_time[gesture_name] = current_time
            return "⬇️ Scroll Down", (0, 255, 0)
        
        elif action == "pause_play":
            pyautogui.press('playpause')
            last_action_time[gesture_name] = current_time
            return "⏯️ Play/Pause", (0, 255, 0)
        
        elif action == "left_click":
            pyautogui.click()
            last_action_time[gesture_name] = current_time
            return "🖱️ Left Click", (0, 255, 0)
        
        elif action == "right_click":
            pyautogui.rightClick()
            last_action_time[gesture_name] = current_time
            return "🖱️ Right Click", (0, 255, 0)
        
        elif action == "next_track":
            pyautogui.press('nexttrack')
            last_action_time[gesture_name] = current_time
            return "⏭️ Next Track", (0, 255, 0)
        
        elif action == "prev_track":
            pyautogui.press('prevtrack')
            last_action_time[gesture_name] = current_time
            return "⏮️ Prev Track", (0, 255, 0)
        
        elif action == "screenshot":
            pyautogui.hotkey('win', 'shift', 's')
            last_action_time[gesture_name] = current_time
            return "📸 Screenshot", (0, 255, 0)
        
        elif action == "minimize_window":
            pyautogui.hotkey('win', 'down')
            last_action_time[gesture_name] = current_time
            return "🔽 Minimize", (0, 255, 0)
        
        elif action == "copy":
            pyautogui.hotkey('ctrl', 'c')
            last_action_time[gesture_name] = current_time
            return "📋 Copy", (0, 255, 0)
        
        elif action == "alt_tab":
            pyautogui.hotkey('alt', 'tab')
            last_action_time[gesture_name] = current_time
            return "🔄 Switch Window", (0, 255, 0)
        
        elif action == "brightness_up":
            # Brightness up (thường dùng Fn key nhưng có thể dùng MonitorBrightnessUp trên Windows)
            pyautogui.press('brightnessup')
            last_action_time[gesture_name] = current_time
            return "🔆 Brightness Up", (0, 255, 0)
        
        else:
            return "Unknown Action", (128, 128, 128)
            
    except Exception as e:
        return f"Error: {str(e)}", (0, 0, 255)

In [30]:
# [MỚI] Khai báo biến trạng thái kéo thả bên ngoài vòng lặp
is_dragging = False

try:
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,  # Chỉ nhận 1 tay để điều khiển chính xác hơn
        min_detection_confidence=0.7,
        min_tracking_confidence=0.5,
    ) as hands:
        print("🎮 Bắt đầu điều khiển bằng cử chỉ!")
        print("📋 Phím tắt:")
        print("  'q' - Thoát")
        print("  'c' - Bật/Tắt điều khiển")
        print("\n🎯 Danh sách cử chỉ:")
        for gesture, action in GESTURE_ACTIONS.items():
            print(f"  {gesture} → {action}")
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)

            frame_h, frame_w, _ = frame.shape
            gesture_name_display = "Unknown"
            action_status = ""
            status_color = (255, 255, 255)

            if result.multi_hand_landmarks:
                hand_landmarks = result.multi_hand_landmarks[0]
                
                # Lấy thông tin trái/phải
                hand_label = "Unknown"
                if result.multi_handedness:
                    hand_label = result.multi_handedness[0].classification[0].label

                # Vẽ landmark
                mp_draw.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_draw.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=3),
                    mp_draw.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                )

                # --- [MỚI] LOGIC KÉO THẢ (DRAG & DROP) ---
                # 1. Lấy tọa độ đầu ngón cái (4) và ngón trỏ (8)
                thumb_tip = hand_landmarks.landmark[4]
                index_tip = hand_landmarks.landmark[8]
                
                # 2. Chuyển đổi sang tọa độ màn hình (để di chuyển chuột)
                screen_w, screen_h = pyautogui.size()
                # Dùng tọa độ ngón trỏ để di chuyển chuột
                mouse_x = int(index_tip.x * screen_w)
                mouse_y = int(index_tip.y * screen_h)
                
                # 3. Tính khoảng cách (Euclidean distance) trong không gian 2D của khung hình
                # Lưu ý: distance này dựa trên tọa độ chuẩn hóa (0.0 - 1.0)
                distance = ((thumb_tip.x - index_tip.x)**2 + (thumb_tip.y - index_tip.y)**2)**0.5
                
                # 4. Ngưỡng xác định chụm tay (cần tinh chỉnh tùy khoảng cách camera)
                PINCH_THRESHOLD = 0.04  

                # 5. Xử lý logic
                if distance < PINCH_THRESHOLD:
                    if not is_dragging:
                        pyautogui.mouseDown()
                        is_dragging = True
                    
                    # Hiển thị trạng thái đang kéo
                    cv2.putText(frame, "✊ DRAGGING", (int(index_tip.x * frame_w), int(index_tip.y * frame_h) - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                else:
                    if is_dragging:
                        pyautogui.mouseUp()
                        is_dragging = False

                # 6. Di chuyển chuột (Chỉ di chuyển khi chế độ Drag đang bật)
                if is_dragging:
                    pyautogui.moveTo(mouse_x, mouse_y)
                
                # --- [HẾT PHẦN LOGIC KÉO THẢ] ---

                # Trích xuất landmark để nhận diện cử chỉ (MLP Model)
                data_point = []
                for lm in hand_landmarks.landmark:
                    data_point.extend([lm.x, lm.y, lm.z])

                if len(data_point) == 63:
                    X = np.array(data_point, dtype=np.float32).reshape(1, -1)
                    X = scaler.transform(X)

                    probs = model.predict(X, verbose=0)[0]
                    proba_buffer.append(probs)
                    probs_smoothed = np.mean(proba_buffer, axis=0)

                    pred_class_idx = np.argmax(probs_smoothed)
                    gesture_name = le.inverse_transform([pred_class_idx])[0]
                    confidence = float(probs_smoothed[pred_class_idx]) * 100

                    if probs_smoothed[pred_class_idx] < UNKNOWN_THRESHOLD:
                        gesture_name_display = "Unknown"
                    else:
                        gesture_name_display = gesture_name
                        # Thực hiện hành động điều khiển (chỉ khi không đang kéo thả để tránh xung đột)
                        if not is_dragging: 
                            action_status, status_color = perform_action(gesture_name_display)

                    # Hiển thị thông tin cử chỉ
                    cx = int(hand_landmarks.landmark[0].x * frame_w)
                    cy = int(hand_landmarks.landmark[0].y * frame_h)
                    cv2.putText(frame, f"{hand_label}: {gesture_name_display} ({confidence:.1f}%)",
                                (cx - 100, cy - 40), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, (0, 255, 255), 2)
                    
                    # Hiển thị trạng thái hành động
                    if action_status:
                        cv2.putText(frame, action_status,
                                    (cx - 100, cy - 10), cv2.FONT_HERSHEY_SIMPLEX,
                                    0.6, status_color, 2)
                else:
                    cv2.putText(frame, "Không đủ landmark",
                                (10, 80),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            else:
                cv2.putText(frame, "Không phát hiện bàn tay", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                # Nếu mất dấu tay khi đang kéo, thả chuột ra để tránh kẹt
                if is_dragging:
                    pyautogui.mouseUp()
                    is_dragging = False

            # Hiển thị trạng thái điều khiển
            control_status = "ON" if action_enabled else "OFF"
            control_color = (0, 255, 0) if action_enabled else (0, 0, 255)
            cv2.putText(frame, f"Control: {control_status}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, control_color, 2)
            cv2.putText(frame, "Press 'c' to toggle | 'q' to quit", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            cv2.imshow("🎮 Gesture Control", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('c'):
                action_enabled = not action_enabled
                status = "BẬT" if action_enabled else "TẮT"
                print(f"🎮 Đã {status} điều khiển cử chỉ")
                
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("✅ Đã thoát chương trình")

🎮 Bắt đầu điều khiển bằng cử chỉ!
📋 Phím tắt:
  'q' - Thoát
  'c' - Bật/Tắt điều khiển

🎯 Danh sách cử chỉ:
  thumb up → volume_up
  fist → volume_down
  peace → scroll_up
  ok → scroll_down
  palm → pause_play
  index up → left_click
  call → right_click
  rock → next_track
  gun sign → prev_track
  L sign → screenshot
  C sign → copy
  cross fingers → alt_tab
  3 fingers up → brightness_up
✅ Đã thoát chương trình
